# Descricao do Projeto: 

Você trabalha em uma startup que vende produtos alimentícios. Você precisa analisar o comportamento do usuário para o aplicativo da empresa.

Primeiro, estude o funil de vendas. Descubra como os usuários chegam à etapa de compra.  usuários realmente chegam a essa etapa? Quantos ficam presos nas fases anteriores? Quais etapas em particular?

Em seguida, veja os resultados do teste A/A/B. (Continue lendo para obter mais informação sobre os testes A/A/B). Os designers gostariam de alterar as fontes de todo o aplicativo, mas os gerentes temem que os usuários achem o novo design intimidador. Eles decidem tomar a decisão com base nos resultados de um teste A/A/B.

Os usuários são divididos em três grupos: dois grupos de controle recebem as fontes antigas e um grupo de teste recebe as novas. Descubra qual conjunto de fontes produz melhores resultados.

A criação de dois grupos A tem certas vantagens. Podemos adaptar um princípio segundo o qual só estaremos confiantes na precisão de nossos testes quando os dois grupos de controle forem semelhantes. Se houver diferenças significativas entre os grupos A, isso pode nos ajudar a descobrir fatores que podem distorcer os resultados. A comparação de grupos de controle também nos informa de quanto tempo e dados precisaremos ao executar outros testes.

Você usará o mesmo conjunto de dados para análise geral e análise A/A/B. Em projetos reais, os experimentos se realizam constantemente. Os analistas estudam a qualidade de um aplicativo usando dados gerais, sem prestar atenção na participação dos usuários em experimentos.

### Descrição dos dados
Cada linha do registro (também referido como log) é uma ação do usuário ou um evento.

EventName — nome do evento
DeviceIDHash — identificador de usuário exclusivo
EventTimestamp — hora do evento
ExpId — número do experimento: 246 e 247 são os grupos de controle, 248 é o grupo de teste


In [ ]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from scipy import stats as st

In [ ]:
df_logs = pd.read_csv('../data/logs_exp_us.csv', sep='\t')
df_logs.columns = df_logs.columns.str.strip()
df_logs.head()

,EventName,DeviceIDHash,EventTimestamp,ExpId
0,MainScreenAppear,4575588528974610257,1564029816,246
1,MainScreenAppear,7416695313311560658,1564053102,246
2,PaymentScreenSuccessful,3518123091307005509,1564054127,248
3,CartScreenAppear,3518123091307005509,1564054127,248
4,PaymentScreenSuccessful,6217807653094995999,1564055322,248


In [ ]:
df_logs = df_logs.rename(
    columns={
        'EventName': 'event_name',
        'DeviceIDHash': 'device_id_hash',
        'EventTimestamp': 'time_stamp',
        'ExpId': 'exp_id'
    }
)

df_logs.head()

,event_name,device_id_hash,time_stamp,exp_id
0,MainScreenAppear,4575588528974610257,1564029816,246
1,MainScreenAppear,7416695313311560658,1564053102,246
2,PaymentScreenSuccessful,3518123091307005509,1564054127,248
3,CartScreenAppear,3518123091307005509,1564054127,248
4,PaymentScreenSuccessful,6217807653094995999,1564055322,248


In [ ]:
df_logs.info()

<class 'pandas.DataFrame'>
RangeIndex: 244126 entries, 0 to 244125
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   event_name      244126 non-null  str  
 1   device_id_hash  244126 non-null  int64
 2   time_stamp      244126 non-null  int64
 3   exp_id          244126 non-null  int64
dtypes: int64(3), str(1)
memory usage: 7.5 MB


In [ ]:
df_logs['time_stamp'] = pd.to_datetime(df_logs['time_stamp'], unit='s')

df_logs['date'] = df_logs['time_stamp'].dt.date
df_logs['hours'] = df_logs['time_stamp'].dt.hour

df_logs.head()

,event_name,device_id_hash,time_stamp,exp_id,date
0,MainScreenAppear,4575588528974610257,2019-07-25 04:43:36,246,2019-07-25
1,MainScreenAppear,7416695313311560658,2019-07-25 11:11:42,246,2019-07-25
2,PaymentScreenSuccessful,3518123091307005509,2019-07-25 11:28:47,248,2019-07-25
3,CartScreenAppear,3518123091307005509,2019-07-25 11:28:47,248,2019-07-25
4,PaymentScreenSuccessful,6217807653094995999,2019-07-25 11:48:42,248,2019-07-25


In [ ]:
print(df_logs.duplicated().sum())

413


In [ ]:
df_logs[df_logs.duplicated(keep=False)].sort_values(
    ['device_id_hash', 'time_stamp']
).head(20)

,event_name,device_id_hash,time_stamp,exp_id,date
130557,MainScreenAppear,33176906322804559,2019-08-04 15:37:55,248,2019-08-04
130558,MainScreenAppear,33176906322804559,2019-08-04 15:37:55,248,2019-08-04
130994,OffersScreenAppear,33176906322804559,2019-08-04 15:49:23,248,2019-08-04
130995,OffersScreenAppear,33176906322804559,2019-08-04 15:49:23,248,2019-08-04
104106,CartScreenAppear,34565258828294726,2019-08-03 18:33:41,248,2019-08-03
104108,CartScreenAppear,34565258828294726,2019-08-03 18:33:41,248,2019-08-03
159301,PaymentScreenSuccessful,54903596035630984,2019-08-05 13:27:52,248,2019-08-05
159302,PaymentScreenSuccessful,54903596035630984,2019-08-05 13:27:52,248,2019-08-05
176812,PaymentScreenSuccessful,77364241990273403,2019-08-05 22:38:52,246,2019-08-05
176813,PaymentScreenSuccessful,77364241990273403,2019-08-05 22:38:52,246,2019-08-05


In [ ]:
df_logs = df_logs.drop_duplicates()
print(df_logs.duplicated().sum())

0


In [ ]:
df_logs.info()

##### Analisando os dados descobrimos 413 duplicados, verificamos que os registros apresentam os mesmos valores de evento, usuario, timestamp e grupo experimental, e por isso foi considerado registros duplicados, decidindo remove-los antes da analise do funil e do experimento. 

### 3. Estude e verifique os dados

- Quantos eventos ficam nos registros?